### Loading Required Modules

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    cohen_kappa_score,
    classification_report,
    log_loss
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping


# Disable warnings
import warnings
def warn(*args, **kwargs):
    pass
warnings.warn = warn

# Load Data
odf = pd.read_csv("Stocks/googl.us.txt")
df = odf.opy(True)

FileNotFoundError: [Errno 2] No such file or directory: 'Stocks/googl.us.txt'

#### Helper Functions

In [ ]:
def create_sequences(data, window_size=60):
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i])
        y.append(data[i])
    return np.array(X), np.array(y)

import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

def plot_results(history, y_true, pred, train_time, params):
    
    # =========================
    # 1. Training Metrics
    # =========================
    plt.figure(figsize=(14,5))

    # MAE
    plt.subplot(1,2,1)
    plt.plot(history.history['mae'], label='Train MAE')
    plt.plot(history.history['val_mae'], label='Val MAE')
    plt.title("MAE over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("MAE")
    plt.legend()

    # MSE
    plt.subplot(1,2,2)
    plt.plot(history.history['mse'], label='Train MSE')
    plt.plot(history.history['val_mse'], label='Val MSE')
    plt.title("MSE over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()

    plt.show()

    # =========================
    # 2. Predictions vs True
    # =========================
    plt.figure(figsize=(12,6))
    plt.plot(y_true, label="True")
    plt.plot(pred, label="Prediction")
    plt.title("Prediction vs Ground Truth")
    plt.legend()
    plt.show()

    # =========================
    # 3. Final Metrics
    # =========================
    mae = mean_absolute_error(y_true, pred)
    mse = mean_squared_error(y_true, pred)

    print(f"Train Time: {train_time:.2f} sec")
    print(f"Parameters: {params}")
    print(f"Test MAE: {mae:.4f}")
    print(f"Test MSE: {mse:.4f}")

## 3. Recurrent Neural Networks (RNNs) — Sequence Modeling

### A. Implement Three Models

## Vanilla RNN

In [ ]:
# Prepare Data
data = df[['Close']].values
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# Create Sequences
window_size = 60
X, y = create_sequences(data_scaled, window_size)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Train Model
early_stop = EarlyStopping(patience=5, restore_best_weights=True)
model = Sequential()
model.add(SimpleRNN(50, return_sequences=False, input_shape=(X_train.shape[1], 1)))
model.add(Dense(1))
model.compile(optimizer="adam", loss="mse", metrics=["mae", "mse"])

start = time.time()
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
train_time = time.time() - start

# Evaluate & Plot
pred = model.predict(X_test)
pred = scaler.inverse_transform(pred)
y_true = scaler.inverse_transform(y_test)
plot_results(history,y_true,pred,train_time,model.count_params())

### LSTM (Long Short-Term Memory)

In [ ]:
# Prepare Data
data = df[['Close']].values
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# Create Sequences
window_size = 60
X, y = create_sequences(data_scaled, window_size)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Train Model
early_stop = EarlyStopping(patience=5, restore_best_weights=True)
model = Sequential()
model.add(LSTM(50, return_sequences=False, input_shape=(X_train.shape[1], 1)))
model.add(Dense(1))
model.compile(optimizer="adam", loss="mse", metrics=["mae", "mse"])

start = time.time()
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
train_time = time.time() - start

# Evaluate & Plot
pred = model.predict(X_test)
pred = scaler.inverse_transform(pred)
y_true = scaler.inverse_transform(y_test)
plot_results(history,y_true,pred,train_time,model.count_params())

### GRU (Gated Recurrent Unit)

In [ ]:
# Prepare Data
data = df[['Close']].values
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# Create Sequences
window_size = 60
X, y = create_sequences(data_scaled, window_size)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Train Model
early_stop = EarlyStopping(patience=5, restore_best_weights=True)
model = Sequential()
model.add(GRU(50, return_sequences=False, input_shape=(X_train.shape[1], 1)))
model.add(Dense(1))
model.compile(optimizer="adam", loss="mse", metrics=["mae", "mse"])

start = time.time()
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
train_time = time.time() - start

# Evaluate & Plot
pred = model.predict(X_test)
pred = scaler.inverse_transform(pred)
y_true = scaler.inverse_transform(y_test)
plot_results(history,y_true,pred,train_time,model.count_params())

## B. Use Core Sequence Modeling Components and Explain

### Sequence Length

### Hidden Size

### One vs multiple recurrent layers

### Bidirectional RNNs

### Dropout between recurrent layers

## C. Discussion Question

## Why are LSTMs and GRUs better than vanilla RNNs for long sequences?

Vanilla RNNs struggle with learning long-term dependencies due to the **vanishing gradient problem**. During backpropagation through time, gradients can shrink exponentially as they are propagated backward across many time steps. This causes earlier information in the sequence to have little to no influence on learning.

LSTMs (Long Short-Term Memory) and GRUs (Gated Recurrent Units) address this limitation by introducing **gating mechanisms** that regulate the flow of information. These architectures are specifically designed to preserve important information over long sequences while discarding irrelevant data.

As a result:

* They can **retain information over longer time spans**
* They are **more stable during training**
* They significantly **outperform vanilla RNNs on long sequences**

---

## Role of Gates in LSTMs and GRUs

Gates are learnable components that control how information flows through the network. They act like **filters**, deciding what information to keep, update, or discard.

### In LSTMs:

There are three main gates:

* **Forget Gate**: Decides what information to remove from memory
* **Input Gate**: Decides what new information to store
* **Output Gate**: Decides what information to output

These gates allow the model to maintain a **cell state**, which acts as a long-term memory.

---

### In GRUs:

GRUs simplify the structure using two gates:

* **Update Gate**: Combines the roles of forget and input gates
* **Reset Gate**: Controls how much past information to forget

---

## How Gates Help with Vanishing Gradients

The key innovation is that LSTMs and GRUs allow gradients to flow more easily through time via controlled pathways:

* The **cell state (in LSTM)** provides a nearly linear path for gradient flow
* Gates **prevent unnecessary transformations**, reducing gradient shrinkage
* Important information can be **preserved across many time steps**

This mitigates the vanishing gradient problem and enables the network to learn **long-term dependencies** effectively.

---

## Summary

* Vanilla RNNs fail on long sequences due to vanishing gradients
* LSTMs and GRUs introduce **gates** to control information flow
* These gates allow the model to:

  * Retain important information
  * Forget irrelevant details
  * Maintain stable gradients

This is why LSTMs and GRUs are the preferred choice for most real-world sequential tasks.